# BTP AI Workshop Day 2
## Grounding with SAP Gen AI Hub + Optional Advanced ReAct Agent

### The Scenario

At BestRun Technologies, the procurement team sources critical components for its smart-IoT security devices from a select group of trusted partners. Ahead of quarterly business-review (QBR) meetings, executives often ask questions such as:

- *"How is Techtronix Components Ltd. performing?"*
- *"What contractual penalties apply if PulseWave Materials Inc. misses a delivery?"*

The answers lie scattered across purchase orders, on-time delivery reports, quality-incident logs, supplier audit scores, and PDF contracts. Manually compiling this information from BestRun Technologies' SharePoint folders can take hours—still risking missed details, such as a late-delivery penalty clause or a recurring pattern of quality incidents with Sigma Electronics Co.

By indexing these documents and leveraging SAP's Generative AI Hub with document grounding, BestRun Technologies can automatically retrieve relevant delivery metrics, quality notes, and contract terms for any supplier, then generate a concise, context-rich briefing for its QBR deck. **This is Part 1 of our workshop: deterministic grounded Q&A.**

But then the CPO asks a harder question:

> *"The Techtronix contract expires in 60 days. Should we renew, renegotiate, or exit?"*

This question cannot be answered with a single retrieval. It requires weighing delivery performance against quality incidents, checking penalty clauses, and synthesizing a recommendation. The next step — which tool to call, which evidence to gather — depends on what we learn along the way. **This is Part 2: when the scenario evolves beyond deterministic workflows, and an agentic solution becomes necessary.**

## Why This Workshop Structure

This workshop enforces a deliberate decision sequence:
1. Define the business decision and evidence required.
2. Choose the simplest architecture that solves it.
3. Use an agent only when deterministic grounding is insufficient.

The scenario evolves naturally: Part 1 handles straightforward QBR questions with grounded retrieval. Part 2 introduces the renewal decision — a multi-criteria problem where the agent adds real value.

### Workshop Roadmap

| Part | Section | ~Time | What You'll Learn |
| --- | --- | --- | --- |
| **1 — Essential** | 🔧 Setup (Steps 1.1–1.3) | 15 min | Install SDK, load config, connect to AI Core |
| | 🔍 Before & After (Steps 1.4–1.6) | 15 min | See the gap between ungrounded and grounded answers |
| | 🏗️ Build: Grounded Q&A (Steps 1.7–1.9) | 30 min | Build and run deterministic grounded queries |
| | 🧭 Decide: Agent or Not? (Step 1.10) | 10 min | Know when grounding is enough vs. when you need an agent |
| | ☕ Checkpoint | 10 min | Reflect + exercise |
| **2 — Advanced** | 🛠️ Agent Tools (Step 2.1) | 10 min | Define specialized tools with evidence outputs |
| | 🔄 ReAct Loop (Steps 2.2–2.3) | 15 min | Build and run a controlled ReAct agent |
| | 👁️ Trace & Approval (Steps 2.4–2.5) | 10 min | Observe agent decisions and human-in-the-loop |

## Business Context and Success Criteria

### How the Scenario Evolves

**Part 1 — Grounded Q&A (Deterministic)**
The procurement team needs quick, accurate answers for the QBR deck:
- *"What are the payment terms in the Techtronix contract?"*
- *"Summarize Techtronix delivery performance."*

These are **single-path questions**: one retrieval, one generation, done. SAP Document Grounding + OrchestrationService handles them cleanly.

**Part 2 — Supplier Renewal Decision (Non-Deterministic)**
Now the CPO escalates: *"Should we renew, renegotiate, or exit the Techtronix contract?"*

Why this requires an agent:
| Dimension | Grounded Q&A | Agent (ReAct) |
| --- | --- | --- |
| Evidence needed | Single domain | Multiple domains (delivery + quality + contract) |
| Next step known? | Yes, fixed | No — depends on intermediate findings |
| Synthesis | Direct answer | Weighted recommendation with trade-offs |

The agent must decide: *Do I have enough delivery data? Should I check quality incidents next? What about contract penalties?* This adaptive reasoning justifies agentic orchestration.

### Data Landscape

| Document | Type | Key Content | Used In |
| --- | --- | --- | --- |
| `techtronix_components_ltd._contract.pdf` | Contract | Payment terms, penalties, termination, governing law | Part 1 + Part 2 |
| `sigma_electronics_co._contract.pdf` | Contract | Competitor benchmark terms | Part 1 Exercise |
| `pulsewave_materials_inc._contract.pdf` | Contract | Competitor benchmark terms | Part 1 Exercise |
| `on_time_delivery_report.txt` | Operational | Delivery performance metrics, delays | Part 1 + Part 2 |
| `quality_incidents_with_contacts.txt` | Quality | Incident logs, severity, resolution | Part 1 + Part 2 |
| `supplier_audits.pdf` | Audit | Supplier quality audit scores | Part 2 |
| `po_with_contacts.txt` | PO data | Purchase order details, contacts | Context |

### What Good Looks Like
By the end of this notebook, participants should be able to:
- Produce grounded answers from enterprise data using SAP services.
- Explain when **not** to use an agent.
- Build one advanced ReAct flow for non-deterministic multi-source decisions.

### SAP BTP Services Used and Why

| BTP Service | Role in This Workshop | Why Not a Generic Alternative? |
| --- | --- | --- |
| **SAP AI Core** | Secure runtime for model deployments | Enterprise auth, resource-group isolation, BTP identity integration |
| **Gen AI Hub SDK** | Pro-code LLM orchestration | Consistent API, version-pinned SDK, SAP support channel |
| **Document Grounding** | Managed retrieval over indexed docs | No custom vector DB ops; managed chunking, embedding, indexing |
| **BTP Object Store (S3)** | Source repository for business documents | Integrated with grounding pipeline; governed access |

This is intentionally **SAP-native** to support production-ready BTP AI adoption.

## Part 1 (Essential): Grounding via SAP Gen AI Hub

### Architecture Focus
This section demonstrates **Grounding** as an enterprise retrieval pattern using SAP-managed services.

```text
Business Question
    -> SAP Document Grounding Retrieval API (Gen AI Hub)
    -> Relevant chunks from indexed documents (BTP Object Store pipeline)
    -> SAP OrchestrationService via Gen AI Hub SDK
    -> Grounded answer with evidence
```

### Why This Matters for BTP Customers
- Keeps security, access, and model execution within SAP AI Core boundaries.
- Uses managed retrieval capabilities instead of custom retrieval infrastructure.
- Provides a repeatable pro-code pattern for enterprise apps.

---
### 🔧 Setup & Configuration

## Learning Objectives and Requirements

### Learning Objectives
By the end of this hands-on, participants will be able to:
- Build grounded enterprise Q&A using SAP Document Grounding and SAP OrchestrationService.
- Distinguish deterministic workflows from non-deterministic agent workflows.
- Implement one advanced ReAct agent loop with guardrails and traceability.

### Requirements
- Shared pre-provisioned workshop environment
- Valid `.env` values for SAP AI Core and grounding repository
- Access to deployed orchestration URL and indexed data repository

### Data Scope for the Use Case
Documents in the repository include supplier contracts, on-time delivery data, quality incidents, and supplier audits.


### Step 1.1 - Install Required Packages

We use SAP's `generative-ai-hub-sdk` as the primary pro-code interface.

Why this SDK:
- Standard SAP-supported path for orchestration and model access.
- Consistent API patterns across workshop and production code.

If your environment is already provisioned, this cell is idempotent.
Restart kernel after install if your platform requests it.


In [ ]:
%pip install generative-ai-hub-sdk==4.12.4 --break-system-packages --quiet
%pip install python-dotenv requests --break-system-packages --quiet

print("Packages are installed.")


### Step 1.2 - Imports and Global Constants

What this step does:
- Imports SAP Gen AI Hub SDK classes for orchestration.
- Imports support libraries for credentials and HTTP calls.
- Defines stable constants to keep workshop behavior deterministic.


In [ ]:
import json
import os
import time
from typing import Any, Dict, List, Tuple

import requests
from dotenv import find_dotenv, load_dotenv

from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.models.template import Template
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage

MODEL_NAME = "anthropic--claude-4.5-sonnet"
TOKEN_TIMEOUT_SECONDS = 30
HTTP_TIMEOUT_SECONDS = 45

print("Imports loaded.")


### Step 1.3 - Load `.env` Configuration

We separate configuration from code using `.env` (required for clean enterprise practices).

Expected environment variable names:
- `AICORE_AUTH_URL`
- `AICORE_CLIENT_ID`
- `AICORE_CLIENT_SECRET`
- `AICORE_BASE_URL`
- `AICORE_RESOURCE_GROUP`
- `ORCH_DEPLOYMENT_URL`
- `DATA_REPOSITORY_ID`

SAP relevance:
- `ORCH_DEPLOYMENT_URL` points to your SAP AI Core orchestration deployment.
- `DATA_REPOSITORY_ID` points to your Document Grounding indexed repository.

> **📌 BTP Design Decision:** We use `OrchestrationService` from the Gen AI Hub SDK (not raw HTTP calls) because it provides a consistent, version-pinned API for LLM orchestration. This means your workshop code patterns transfer directly to production CAP applications.

In [ ]:
dotenv_path = find_dotenv()
load_dotenv(dotenv_path=dotenv_path, override=True)

AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
ORCH_DEPLOYMENT_URL = os.getenv("ORCH_DEPLOYMENT_URL")
DATA_REPOSITORY_ID = os.getenv("DATA_REPOSITORY_ID")

required = {
    "AICORE_AUTH_URL": AICORE_AUTH_URL,
    "AICORE_CLIENT_ID": AICORE_CLIENT_ID,
    "AICORE_CLIENT_SECRET": AICORE_CLIENT_SECRET,
    "AICORE_BASE_URL": AICORE_BASE_URL,
    "AICORE_RESOURCE_GROUP": AICORE_RESOURCE_GROUP,
    "ORCH_DEPLOYMENT_URL": ORCH_DEPLOYMENT_URL,
    "DATA_REPOSITORY_ID": DATA_REPOSITORY_ID,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise EnvironmentError(f"Missing required .env variables: {missing}")

orchestration_service = OrchestrationService(api_url=ORCH_DEPLOYMENT_URL)

print(f"Loaded .env from: {dotenv_path}")
print(f"Resource group: {AICORE_RESOURCE_GROUP}")
print(f"Deployment ID: {ORCH_DEPLOYMENT_URL.split('/')[-1]}")
print(f"Data repository ID: {DATA_REPOSITORY_ID}")


---
### 🔍 Before vs. After: The Grounding Effect

### Step 1.4 - Quick Connection Check (No Grounding Yet)

Purpose:
- Validate that SAP AI Core orchestration endpoint is reachable.
- Show baseline behavior **without** business context.

This creates an explicit before/after comparison so participants see the value of Grounding.

In [ ]:
baseline_response = orchestration_service.run(
    config=OrchestrationConfig(
        template=Template(messages=[
            SystemMessage("You are a procurement analyst."),
            UserMessage("What are the payment terms in the Techtronix contract?")
        ]),
        llm=LLM(name=MODEL_NAME)
    )
)

print("Baseline response (no grounding):")
print("-" * 70)
print(baseline_response.orchestration_result.choices[0].message.content)
print("-" * 70)


### Step 1.5 - Grounding Retrieval Helper

This helper calls SAP Document Grounding retrieval directly:
- Endpoint: `/lm/document-grounding/retrieval/search`
- Auth: AI Core OAuth token (client credentials)
- Scope: your configured resource group and repository

Why we use this in the workshop:
- Participants can see retrieval mechanics and provenance.
- It keeps the flow transparent and debuggable for learning.

Return values:
1. Formatted context for prompt augmentation
2. Source list for evidence traceability
3. Chunk count for observability

> **📌 BTP Design Decision:** We call the Document Grounding API directly (not build our own RAG pipeline) because SAP manages the vector index, chunking strategy, and embedding model. This means zero infrastructure for the workshop participant — and a production-ready retrieval layer for customers.

In [ ]:
_token_cache: Dict[str, Any] = {"access_token": None, "expires_at": 0.0}


def _get_ai_core_token() -> str:
    now = time.time()
    if _token_cache["access_token"] and now < _token_cache["expires_at"] - 30:
        return _token_cache["access_token"]

    token_resp = requests.post(
        f"{AICORE_AUTH_URL}/oauth/token",
        data={"grant_type": "client_credentials"},
        auth=(AICORE_CLIENT_ID, AICORE_CLIENT_SECRET),
        timeout=TOKEN_TIMEOUT_SECONDS,
    )
    token_resp.raise_for_status()
    payload = token_resp.json()

    if "access_token" not in payload:
        raise RuntimeError(f"Token response did not contain access_token. Payload: {payload}")

    _token_cache["access_token"] = payload["access_token"]
    _token_cache["expires_at"] = now + int(payload.get("expires_in", 600))
    return _token_cache["access_token"]


def retrieve_grounding_context(query: str, max_chunks: int = 6) -> Tuple[str, List[str], int]:
    token = _get_ai_core_token()

    response = requests.post(
        f"{AICORE_BASE_URL}/lm/document-grounding/retrieval/search",
        json={
            "query": query,
            "filters": [{
                "id": "supplier_docs",
                "dataRepositories": [DATA_REPOSITORY_ID],
                "dataRepositoryType": "vector"
            }],
            "maxChunkCount": max_chunks
        },
        headers={
            "Authorization": f"Bearer {token}",
            "AI-Resource-Group": AICORE_RESOURCE_GROUP
        },
        timeout=HTTP_TIMEOUT_SECONDS,
    )
    response.raise_for_status()

    payload = response.json()
    chunks: List[str] = []
    sources: List[str] = []

    for filter_result in payload.get("results", []):
        for repo_result in filter_result.get("results", []):
            data_repo = repo_result.get("dataRepository", {})
            for doc in data_repo.get("documents", []):
                metadata = doc.get("metadata", [])
                source = next(
                    (m.get("value", ["unknown"])[0] for m in metadata if m.get("key") == "id"),
                    "unknown"
                )
                for chunk in doc.get("chunks", []):
                    content = (chunk.get("content") or "").strip()
                    if content:
                        chunks.append(f"[Source: {source}]\n{content}")
                        sources.append(source)

    if not chunks:
        return "No relevant grounded context was retrieved.", [], 0

    context = "\n\n---\n\n".join(chunks)
    unique_sources = sorted(set(sources))
    return context, unique_sources, len(chunks)


print("Grounding retrieval helper is ready.")


### Step 1.6 - Inspect Retrieved Grounding Chunks

Run this to inspect what SAP Grounding actually returns for a real procurement query.

Learning objective:
- Verify that retrieval returns enterprise-specific content before LLM generation.


In [ ]:
sample_query = "Techtronix delivery performance and contract penalty terms"
context, source_list, chunk_count = retrieve_grounding_context(sample_query, max_chunks=5)

print(f"Query: {sample_query}")
print(f"Retrieved chunks: {chunk_count}")
print(f"Sources: {source_list}")
print("Preview:")
print("-" * 70)
print(context[:1800])
print("-" * 70)


---
### 🏗️ Build: Deterministic Grounded Q&A

### Step 1.7 - Deterministic Grounded Query Function

This function is your standard **non-agent** enterprise pattern:
1. retrieve grounded context from SAP Grounding
2. augment the prompt
3. generate answer through SAP OrchestrationService

Use this when workflow is fixed and deterministic.

In [ ]:
def grounded_query(question: str, system_role: str = "You are a procurement analyst.", max_chunks: int = 6) -> Dict[str, Any]:
    context, sources, chunk_count = retrieve_grounding_context(question, max_chunks=max_chunks)

    response = orchestration_service.run(
        config=OrchestrationConfig(
            template=Template(messages=[
                SystemMessage(system_role),
                UserMessage(
                    "Use only the grounded context below. If evidence is missing, say so.\n\n"
                    f"Grounded context:\n{context}\n\n"
                    f"Question: {question}"
                ),
            ]),
            llm=LLM(name=MODEL_NAME),
        )
    )

    answer = response.orchestration_result.choices[0].message.content
    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "chunk_count": chunk_count,
    }


print("grounded_query is ready.")


### Step 1.8 - Run Deterministic Grounded Examples

> **🎯 What We're Really Asking the AI:**
> These aren't generic questions — they're the exact questions a procurement analyst would ask during a supplier review. Each one maps to a real business decision point:
> - *"What are the payment and penalty terms?"* → Can we negotiate better terms at renewal?
> - *"Summarize delivery performance"* → Is the supplier reliable enough to keep?

These examples are intentionally single-path questions where agent orchestration is unnecessary.

This reinforces the architecture principle: **if one retrieval + one generation solves it, do not add agent complexity.**

In [ ]:
essential_questions = [
    "What are the payment and late delivery penalty terms in the Techtronix contract?",
    "Summarize Techtronix delivery performance based on available reports.",
]

for q in essential_questions:
    result = grounded_query(q)
    print("=" * 80)
    print(f"Question: {result['question']}")
    print(f"Chunk count: {result['chunk_count']}")
    print(f"Sources: {result['sources']}")
    print("Answer:")
    print(result["answer"])


### Step 1.9 - Exercise (Essential)

Try these deterministic questions with `grounded_query()`:
- What is the Techtronix contract termination notice period?
- What quality issues are recorded for Techtronix?
- Compare late delivery penalty clauses across Techtronix, Sigma, and PulseWave.

Expected participant outcome:
- grounded answers
- visible source references
- clear understanding of deterministic grounding pattern


In [ ]:
your_question = "What is the Techtronix contract termination notice period?"
exercise_result = grounded_query(your_question)

print(f"Question: {your_question}")
print(f"Sources used: {exercise_result['sources']}")
print(exercise_result["answer"])


---
### 🧭 Decide: When to Use an Agent

### Step 1.10 - Decision Framework: When to Use Agent vs Not

This is the key mindset shift for tool-first participants.

| Dimension | Grounded Q&A (`grounded_query`) | Agent (ReAct) |
| --- | --- | --- |
| **When to use** | Single question, fixed process | Multi-criteria, adaptive reasoning |
| **BTP services** | Grounding + Orchestration | Same + tool layer + trace storage |
| **Complexity** | Low | Medium-High |
| **Latency** | 1 retrieval + 1 LLM call | N tool calls + N LLM calls |
| **Governance need** | Low (answer + sources) | High (trace + approval gate) |
| **Example** | "What is the penalty clause?" | "Should we renew this supplier?" |

Use `grounded_query()` when:
- single business question
- fixed process
- no branching tool decisions needed

Use an agent when:
- intermediate findings change next actions
- multiple specialized tools must be selected dynamically
- decision quality depends on iterative evidence synthesis

In [ ]:
def decide_pattern(task_description: str) -> str:
    text = task_description.lower()
    agent_signals = [
        "recommend",
        "should we",
        "compare",
        "trade-off",
        "multiple",
        "if",
        "decision",
        "renew",
    ]
    matched = [s for s in agent_signals if s in text]
    if len(matched) >= 2:
        return f"Agent recommended. Signals: {matched}"
    return "Deterministic grounded_query recommended."

examples = [
    "What is the late delivery penalty for Techtronix?",
    "Should we renew Techtronix based on delivery, quality, and contract risk?",
]
for e in examples:
    print(f"Task: {e}")
    print(f"Decision: {decide_pattern(e)}")
    print("-" * 70)


---

> ✅ **Checkpoint — Part 1 Complete**
>
> You now have a working grounded Q&A pipeline. You can answer single business questions using SAP Document Grounding + OrchestrationService — and you understand *when* this deterministic pattern is sufficient.
>
> **In Part 2** (optional advanced), we move to multi-step, non-deterministic decisions where an agent adds real value: the supplier renewal recommendation.

## Part 2 (Optional Advanced): ReAct Agent on SAP AI Core

This section is optional advanced content.

Goal: demonstrate a pro-code ReAct pattern on SAP services for non-deterministic business decisions.

### Top 5 Agent Architecture Best Practices in This Section

| # | Best Practice | How It's Applied Here |
| --- | --- | --- |
| 1 | **Decision-first architecture** | Use agent only for non-deterministic tasks |
| 2 | **Strict tool contracts** | Typed inputs and controlled outputs |
| 3 | **Grounded evidence and provenance** | Every tool result includes sources |
| 4 | **Guardrails and structured outputs** | JSON schema instead of fragile free text parsing |
| 5 | **Observability and human oversight** | Step traces plus approval checkpoint |

### Step 2.1 - Define Agent Tools with Evidence Outputs

Each tool wraps a focused business capability and uses SAP services under the hood:
- SAP Document Grounding retrieval for evidence acquisition
- SAP OrchestrationService for role-specific synthesis

Why this is important for BTP adoption:
- modular design with clear responsibilities
- evidence-backed outputs suitable for enterprise review
- SAP-native runtime path from prototype to production

> **📌 BTP Design Decision:** Each agent tool wraps SAP services (not generic OpenAI calls) because in production, the entire execution stays within SAP AI Core boundaries — enterprise auth, resource-group isolation, and audit trail included.

In [ ]:
def _summarize_with_role(system_role: str, question: str, context: str) -> str:
    response = orchestration_service.run(
        config=OrchestrationConfig(
            template=Template(messages=[
                SystemMessage(system_role),
                UserMessage(
                    "Use only the grounded context below and be concise.\n\n"
                    f"Grounded context:\n{context}\n\nQuestion: {question}"
                ),
            ]),
            llm=LLM(name=MODEL_NAME),
        )
    )
    return response.orchestration_result.choices[0].message.content


def query_delivery_performance(supplier_name: str) -> Dict[str, Any]:
    query = f"{supplier_name} delivery report on-time late penalty"
    context, sources, chunk_count = retrieve_grounding_context(query, max_chunks=5)
    summary = _summarize_with_role(
        "You are a supply chain analyst. Return 3-5 bullet points.",
        f"Summarize delivery performance for {supplier_name}.",
        context,
    )
    return {
        "tool": "query_delivery_performance",
        "supplier_name": supplier_name,
        "summary": summary,
        "sources": sources,
        "chunk_count": chunk_count,
    }


def query_quality_incidents(supplier_name: str) -> Dict[str, Any]:
    query = f"{supplier_name} quality incidents audit score defects"
    context, sources, chunk_count = retrieve_grounding_context(query, max_chunks=5)
    summary = _summarize_with_role(
        "You are a quality analyst. Return 3-5 bullet points.",
        f"Summarize quality incidents and quality score for {supplier_name}.",
        context,
    )
    return {
        "tool": "query_quality_incidents",
        "supplier_name": supplier_name,
        "summary": summary,
        "sources": sources,
        "chunk_count": chunk_count,
    }


def query_contract_terms(supplier_name: str, topic: str) -> Dict[str, Any]:
    query = f"{supplier_name} contract {topic} clauses"
    context, sources, chunk_count = retrieve_grounding_context(query, max_chunks=5)
    summary = _summarize_with_role(
        "You are a contract analyst. Cite exact clauses when present.",
        f"Extract contract terms for {supplier_name} about: {topic}.",
        context,
    )
    return {
        "tool": "query_contract_terms",
        "supplier_name": supplier_name,
        "topic": topic,
        "summary": summary,
        "sources": sources,
        "chunk_count": chunk_count,
    }


TOOLS = {
    "query_delivery_performance": query_delivery_performance,
    "query_quality_incidents": query_quality_incidents,
    "query_contract_terms": query_contract_terms,
}

print(f"Agent tools ready: {list(TOOLS.keys())}")


### Step 2.2 - ReAct Prompt and Structured Action Parser

We implement a controlled ReAct protocol using strict JSON outputs.

Why JSON format:
- more reliable parsing in workshop conditions
- easier validation and error handling
- closer to production-grade agent orchestration

### ReAct Flow Diagram

```text
User decision question
        |
        v
+-------------------------------+
| ReAct controller (LLM)        |
| chooses next action           |
+-------------------------------+
        |
        | ACTION + ARGS (JSON)
        v
+-------------------------------+
| Tool layer                    |
| - delivery performance tool   |
| - quality incidents tool      |
| - contract terms tool         |
+-------------------------------+
        |
        | evidence + sources
        v
+-------------------------------+
| Observation memory            |
| append tool result to context |
+-------------------------------+
        |
        | loop until enough evidence
        v
+-------------------------------+
| FINAL recommendation          |
| + confidence + trace          |
+-------------------------------+
        |
        v
Human approval gate (HITL)
```


In [ ]:
AGENT_SYSTEM_PROMPT = """
You are a procurement decision agent for BestRun Technologies.

Rules:
- You must use tools. Do not answer from prior knowledge.
- Think step by step and choose tools based on evidence gaps.
- Return ONLY valid JSON in one of these forms.

For tool calls:
{
  "type": "action",
  "thought": "short reason",
  "tool": "query_delivery_performance | query_quality_incidents | query_contract_terms",
  "args": {"supplier_name": "Techtronix Components Ltd."}
}

For final answer:
{
  "type": "final",
  "thought": "short synthesis",
  "answer": "final recommendation",
  "confidence": "low|medium|high"
}
""".strip()


def parse_agent_json(text: str) -> Dict[str, Any]:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return {"type": "retry", "error": "No JSON object found."}

    snippet = text[start:end + 1]
    try:
        payload = json.loads(snippet)
    except json.JSONDecodeError as exc:
        return {"type": "retry", "error": f"Invalid JSON: {exc}"}

    if payload.get("type") not in {"action", "final"}:
        return {"type": "retry", "error": "Missing or invalid 'type'."}
    return payload


print("Agent parser ready.")


### Step 2.3 - Run ReAct Agent (with Trace + Approval Gate)

This runtime loop demonstrates enterprise controls:
- minimum tool-usage guardrail
- schema validation and retry behavior
- step-level observability trace
- explicit human approval state before operational action


In [ ]:
def run_react_agent(
    question: str,
    min_tools: int = 2,
    max_steps: int = 6,
    require_human_approval: bool = True,
) -> Dict[str, Any]:
    messages = [SystemMessage(AGENT_SYSTEM_PROMPT), UserMessage(question)]
    tool_history: List[Dict[str, Any]] = []
    trace: List[Dict[str, Any]] = []

    print("=" * 80)
    print("REACT AGENT START")
    print(f"Question: {question}")
    print("=" * 80)

    for step in range(1, max_steps + 1):
        step_start = time.time()
        response = orchestration_service.run(
            config=OrchestrationConfig(
                template=Template(messages=messages),
                llm=LLM(name=MODEL_NAME),
            )
        )
        raw = response.orchestration_result.choices[0].message.content
        parsed = parse_agent_json(raw)

        elapsed_ms = int((time.time() - step_start) * 1000)

        if parsed.get("type") == "action":
            tool_name = parsed.get("tool")
            args = parsed.get("args", {})

            if tool_name not in TOOLS:
                tool_result = {"error": f"Unknown tool: {tool_name}"}
            else:
                try:
                    tool_result = TOOLS[tool_name](**args)
                    tool_history.append({"tool": tool_name, "args": args})
                except Exception as exc:
                    tool_result = {"error": f"Tool failed: {exc}"}

            trace.append({
                "step": step,
                "type": "action",
                "thought": parsed.get("thought", ""),
                "tool": tool_name,
                "args": args,
                "latency_ms": elapsed_ms,
                "result_sources": tool_result.get("sources", []),
            })

            print(f"Step {step}: ACTION -> {tool_name}({args})")

            messages.append(UserMessage(
                "Tool result (JSON):\n"
                f"{json.dumps(tool_result, ensure_ascii=True)}\n\n"
                f"Tools used so far: {len(tool_history)}."
            ))
            continue

        if parsed.get("type") == "final":
            if len(tool_history) < min_tools:
                messages.append(UserMessage(
                    f"You used {len(tool_history)} tool(s). Use at least {min_tools} tools before final answer."
                ))
                trace.append({
                    "step": step,
                    "type": "guardrail",
                    "thought": "Minimum tool count not met.",
                    "tool": None,
                    "args": None,
                    "latency_ms": elapsed_ms,
                    "result_sources": [],
                })
                print(f"Step {step}: GUARDRAIL -> requires at least {min_tools} tools")
                continue

            final_answer = parsed.get("answer", "")
            confidence = parsed.get("confidence", "unknown")
            trace.append({
                "step": step,
                "type": "final",
                "thought": parsed.get("thought", ""),
                "tool": None,
                "args": None,
                "latency_ms": elapsed_ms,
                "result_sources": [],
            })

            print(f"Step {step}: FINAL reached (confidence={confidence})")

            decision = "approved"
            if require_human_approval:
                decision = "pending_human_approval"

            return {
                "question": question,
                "answer": final_answer,
                "confidence": confidence,
                "tool_history": tool_history,
                "trace": trace,
                "approval_state": decision,
            }

        trace.append({
            "step": step,
            "type": "retry",
            "thought": parsed.get("error", "Parser retry"),
            "tool": None,
            "args": None,
            "latency_ms": elapsed_ms,
            "result_sources": [],
        })
        messages.append(UserMessage("Return only one valid JSON object in the required schema."))
        print(f"Step {step}: RETRY -> {parsed.get('error', 'format error')}")

    return {
        "question": question,
        "answer": "Agent reached max steps without final answer.",
        "confidence": "low",
        "tool_history": tool_history,
        "trace": trace,
        "approval_state": "incomplete",
    }


print("run_react_agent is ready.")


### Step 2.4 - Run One Advanced Agent Scenario

Single advanced scenario for this workshop:
- Should BestRun renew Techtronix based on delivery, quality, and contract evidence?

This is intentionally non-deterministic and requires selective multi-tool reasoning.


In [ ]:
agent_result = run_react_agent(
    question=(
        "Should BestRun Technologies renew the Techtronix Components Ltd. contract? "
        "Use delivery performance, quality incidents, and contract terms. "
        "Provide a clear recommendation and rationale."
    ),
    min_tools=2,
    max_steps=6,
    require_human_approval=True,
)

print("\nFinal recommendation:")
print("-" * 80)
print(agent_result["answer"])
print("-" * 80)
print(f"Approval state: {agent_result['approval_state']}")
print(f"Tools used: {agent_result['tool_history']}")


### Step 2.5 - Observability Trace (Best Practice)

Architect takeaway:
- enterprise agents require transparent step traces
- trace data supports governance, troubleshooting, and stakeholder trust


In [ ]:
for row in agent_result["trace"]:
    print(
        f"step={row['step']} | type={row['type']} | tool={row['tool']} | "
        f"latency_ms={row['latency_ms']} | thought={row['thought'][:100]}"
    )


## Summary and Architecture Takeaways

This pro-code solution intentionally separates two architectural modes:
- **Deterministic grounding workflow** for fixed, single-path business questions.
- **Agentic workflow** for non-deterministic, multi-step business decisions.

### 1) Grounding Takeaways

Grounding is the **default enterprise pattern** when the task is clear and the path is fixed.

Architecture pattern:
1. Retrieve evidence from SAP Document Grounding repository.
2. Augment prompt with retrieved context.
3. Generate answer via SAP orchestration runtime.

| Dimension | Assessment |
| --- | --- |
| **When to choose** | Single question, fixed process, low orchestration complexity |
| **Pros** | Simple, transparent, lower operational overhead, easier onboarding |
| **Limits** | Limited adaptive reasoning, weaker fit for multi-criteria recommendations |

SAP BTP value mapping:
- **SAP Document Grounding (Gen AI Hub)**: managed retrieval over indexed enterprise documents.
- **SAP AI Core**: secure model runtime and deployment abstraction.
- **Gen AI Hub SDK**: stable pro-code API for repeatable implementation.
- **BTP Object Store (S3)**: enterprise data source integrated into the grounding pipeline.

### 2) Agentic Takeaways

Agentic design is justified **only** when workflow steps cannot be fully predetermined.

In this workshop, the renewal decision is non-deterministic because the system must:
- decide which evidence domain to query first,
- adapt next tool choice based on intermediate findings,
- synthesize delivery, quality, and contract evidence into one recommendation.

Core architecture decisions used:

| # | Decision | Purpose |
| --- | --- | --- |
| 1 | **Tool specialization** | Separate delivery, quality, and contract tools |
| 2 | **Structured control protocol** | JSON action/final schema for robust orchestration |
| 3 | **Guardrails** | Minimum tool-use threshold and retry on schema violations |
| 4 | **Provenance by design** | Each tool result includes sources and chunk counts |
| 5 | **Observability and governance** | Per-step trace and explicit human approval state |

Key design trade-offs:

| Trade-off | Benefit | Cost |
| --- | --- | --- |
| Agent flexibility vs reliability | Increases decision quality for complex tasks | Requires stricter controls (schema, retries, safeguards) |
| Tool granularity vs maintenance | Finer-grained tools improve controllability | Increases prompt/tool contract maintenance |
| Autonomy vs governance | Higher autonomy accelerates analysis | Business-critical decisions still require HITL checkpoints |
| Lightweight ReAct vs frameworks | Improves teaching transparency | Framework-based orchestration better for production scale |

SAP BTP value mapping for agentic solutions:
- **SAP AI Core**: enterprise runtime boundary, resource-group control, and model endpoint management.
- **Gen AI Hub SDK OrchestrationService**: consistent LLM invocation layer for controller and tools.
- **Document Grounding APIs**: shared retrieval backbone for all tools, preserving evidence consistency.
- **BTP platform services**: identity, security, and operational alignment for production-readiness.

### Final Architectural Principle

Start with the simplest SAP-native pattern that solves the business problem:
- use **grounding-only** for deterministic Q&A,
- add **agentic orchestration** only when business decisions require adaptive, multi-step evidence reasoning.

This is the practical path from pilot demos to production-grade BTP AI solutions.

---

## 📖 Architecture Discussion — See Separate Guide

The full architecture appendix has been moved to a dedicated document for the architecture discussion segment:

**[`appendix_architecture.md`](appendix_architecture.md)**

| Appendix | Topic |
|----------|-------|
| **A** | From Notebook Prototype to CAP Application |
| **B** | End-to-End Data Integration from SAP S/4HANA |
| **C** | Connecting Your Agent to Business Applications (Go-Live) |

> **Facilitator note:** Use `appendix_architecture.md` for the architecture discussion after the hands-on coding exercise. Estimated time: 30–45 minutes.
